# Calibration Notebook

**Run order:** After Notebook A v3 and Notebook B v3. Before re-running Notebook D (gold_person_scores).

**Purpose:** Review signal fire rates, raw score distributions, per-signal contributions,
score independence, and story-ready threshold — before committing base scores.

## Cells
1. Signal fire rates with pass/fail guidance
2. Raw score distributions (percentiles) for each scoring dimension
3. Per-signal average score contribution
4. Score independence cross-tab (completeness vs evidence vs story)
5. Story-ready threshold calibration
6. Top candidates spot-check — review highest scorers per dimension

## What to do with the results
- **Fire rates**: flag anything outside the guidance ranges (see Cell 1 comments)
- **Distributions**: look for ceiling/floor effects — ideal spread is roughly normal,
  not bunched at 0 or 100
- **Contributions**: signals contributing < 1 point on average may need a higher base_score
  or their guard conditions reviewed
- **Independence**: completeness and evidence should not correlate perfectly — if they do,
  some signals may be double-counting the same underlying gap
- **Story threshold**: pick the threshold that gives ~50 story-ready people across the tree
  (the 50 Stories goal), weighted towards direct ancestors

Once you are happy with the results, update base scores in Notebook A v3 and re-run
Notebook B v3, then this notebook, then Notebook D.


In [0]:
%sql
-- ============================================================
-- CELL 1: Signal fire rates
--
-- Replaces: Notebook B v3 Cell 3 (identical query — see deduplication note)
--
-- Guidance ranges (v3 baseline — tune after first run):
--
-- COMPLETENESS signals:
--   SIGNAL_NO_BIRTH_RECORDED        expect 20-40%  (many pre-civil-reg ancestors)
--   SIGNAL_NO_DEATH_RECORDED        expect 20-40%
--   SIGNAL_MISSING_PARENT           expect 40-60%  (end-of-line profiles)
--   SIGNAL_NO_MARRIAGES             expect 15-30%
--   SIGNAL_MISSING_CENSUS_COVERAGE  expect 40-70%  (census gaps are common)
--   SIGNAL_UNCOVERED_SOURCES        was 15-25% at proximity<=2; expect 25-45% now widened
--   SIGNAL_DOCS_NOT_TRANSCRIBED     expect 5-20%   (OCR coverage partial)
--   SIGNAL_LATE_LIFE_GAP            expect 15-35%
--   SIGNAL_EARLY_LIFE_ONLY          expect 10-25%
--   SIGNAL_CHILD_GAPS               expect 10-25%
--   SIGNAL_UNCONFIRMED_MILITARY     expect 10-20%  (1880-1928 birth range subset)
--   SIGNAL_MISSING_OCCUPATION       expect 20-40%  (many males no occ recorded)
--   SIGNAL_MISSING_BURIAL           expect 30-60%  (burial events often not in tree)
--   SIGNAL_NO_RESIDENCE             FLAG if > 70%  — may indicate RESI event type
--                                   name mismatch; check gold_person_event_timeline
--   SIGNAL_NO_CHILDREN              expect < 5%    (proximity <= 1 guard)
--
-- EVIDENCE signals:
--   SIGNAL_NO_DOCUMENTS_AT_ALL      expect 60-85%  (OCR coverage still partial)
--   SIGNAL_FACT_CONFLICT            expect < 5%    (Cuthbertson only currently)
--   SIGNAL_UNRESOLVED_CONFLICT      expect < 5%    (subset of FACT_CONFLICT)
--   SIGNAL_TRANSCRIPT_ONLY_FACTS    expect < 10%
--   SIGNAL_VERY_LOW_EVIDENCE_DENSITY expect ~8%
--   SIGNAL_LOW_EVIDENCE_DENSITY     expect ~77%    (was 99.3% at < 1.2)
--   SIGNAL_SINGLE_SOURCE_DEPENDENCE expect 10-25%
--   SIGNAL_UNSOURCED_FAMILY_EVENTS  expect 20-40%
--   SIGNAL_IMPRECISE_DATES          expect 20-40%  (reduced from 67% post-1837 guard)
--   SIGNAL_IMPRECISE_PLACES         expect 20-50%
--   SIGNAL_INCOMPLETE_NAME          expect 5-15%
--
-- NARRATIVE signals:
--   SIGNAL_MIGRANT                  expect 5-15%
--   SIGNAL_CONFIRMED_MILITARY       expect < 5%    (rare; Cuthbertson only initially)
--   SIGNAL_MILITARY                 expect < 5%
--   SIGNAL_YOUNG_DEATH              expect 5-15%
--   SIGNAL_LARGE_FAMILY             expect < 10%
--   SIGNAL_MULTIPLE_SPOUSES         expect 3-8%
--   SIGNAL_NEWSPAPER_MENTION        expect < 2%    (very rare)
--   SIGNAL_WILL_OR_PROBATE          expect < 5%
--   SIGNAL_STORY_WRITTEN            expect < 1%    (only 1 story written so far)
--   SIGNAL_TRANSCRIPT_RICH          expect < 10%   (OCR coverage partial)
--   SIGNAL_VARIED_OCCUPATIONS       expect 5-15%
--   SIGNAL_HIGH_FAMILY_PAYOFF       expect 20-35%
--   SIGNAL_POSSIBLE_RESIDENCE       expect 20-40%
--   SIGNAL_POSSIBLE_MARRIAGE        expect 3-8%
--   SIGNAL_POSSIBLE_CHILDREN        expect 5-15%
-- ============================================================

SELECT
  signal_code,
  COUNT(*)                                          AS fire_count,
  ROUND(COUNT(*) * 100.0 / MAX(total.cnt), 1)       AS fire_pct,
  -- Quick visual: flag signals that may need attention
  CASE
    WHEN signal_code = 'SIGNAL_NO_RESIDENCE'
      AND COUNT(*) * 100.0 / MAX(total.cnt) > 70  THEN '⚠ check RESI event type'
    WHEN signal_code = 'SIGNAL_LOW_EVIDENCE_DENSITY'
      AND COUNT(*) * 100.0 / MAX(total.cnt) > 95  THEN '⚠ threshold may be too low'
    WHEN signal_code IN ('SIGNAL_UNRESOLVED_CONFLICT','SIGNAL_FACT_CONFLICT',
                          'SIGNAL_TRANSCRIPT_ONLY_FACTS','SIGNAL_CONFIRMED_MILITARY',
                          'SIGNAL_TRANSCRIPT_RICH')
      AND COUNT(*) * 100.0 / MAX(total.cnt) > 20  THEN '⚠ unexpectedly high — check OCR join'
    WHEN COUNT(*) = 0 THEN '⚠ zero fires — check signal logic'
    ELSE 'ok'
  END AS flag
FROM genealogy.gold_research_person_signals_pivoted
CROSS JOIN (SELECT COUNT(DISTINCT person_gedcom_id) AS cnt
            FROM genealogy.gold_research_person_signals) total
GROUP BY signal_code
ORDER BY fire_pct DESC;


In [0]:
%sql
-- ============================================================
-- CELL 2: Raw score distributions
--
-- Shows the distribution of raw scores BEFORE percentile conversion
-- in gold_person_scores. Run this to check for ceiling/floor effects.
--
-- Ideal outcome: each dimension spreads across a reasonable range.
-- Red flags:
--   > 50% of people at raw score 0 for a dimension  → add more signals or raise weights
--   > 50% of people within 10 points of the max     → signals saturating; add harder ones
--   completeness_risk and evidence_fragility highly correlated → signals double-counting
--
-- These raw scores come directly from the priority views and are used as
-- inputs to gold_person_scores. If those views haven't been refreshed
-- since Notebook B v3 ran, re-run them first.
-- ============================================================

-- Completeness risk distribution
SELECT
  'completeness_risk' AS dimension,
  MIN(completeness_risk_score)                   AS min_score,
  ROUND(PERCENTILE(completeness_risk_score, 0.10), 1) AS p10,
  ROUND(PERCENTILE(completeness_risk_score, 0.25), 1) AS p25,
  ROUND(PERCENTILE(completeness_risk_score, 0.50), 1) AS median,
  ROUND(PERCENTILE(completeness_risk_score, 0.75), 1) AS p75,
  ROUND(PERCENTILE(completeness_risk_score, 0.90), 1) AS p90,
  MAX(completeness_risk_score)                   AS max_score,
  ROUND(AVG(completeness_risk_score), 1)         AS mean_score,
  COUNT(*)                                       AS total_people
FROM genealogy.gold_person_integrity_priority

UNION ALL

-- Evidence fragility distribution
SELECT
  'evidence_fragility',
  MIN(evidence_fragility_score),
  ROUND(PERCENTILE(evidence_fragility_score, 0.10), 1),
  ROUND(PERCENTILE(evidence_fragility_score, 0.25), 1),
  ROUND(PERCENTILE(evidence_fragility_score, 0.50), 1),
  ROUND(PERCENTILE(evidence_fragility_score, 0.75), 1),
  ROUND(PERCENTILE(evidence_fragility_score, 0.90), 1),
  MAX(evidence_fragility_score),
  ROUND(AVG(evidence_fragility_score), 1),
  COUNT(*)
FROM genealogy.gold_person_integrity_priority

UNION ALL

-- Narrative priority distribution
SELECT
  'narrative_priority',
  MIN(narrative_priority_score),
  ROUND(PERCENTILE(narrative_priority_score, 0.10), 1),
  ROUND(PERCENTILE(narrative_priority_score, 0.25), 1),
  ROUND(PERCENTILE(narrative_priority_score, 0.50), 1),
  ROUND(PERCENTILE(narrative_priority_score, 0.75), 1),
  ROUND(PERCENTILE(narrative_priority_score, 0.90), 1),
  MAX(narrative_priority_score),
  ROUND(AVG(narrative_priority_score), 1),
  COUNT(*)
FROM genealogy.gold_person_narrative_priority

ORDER BY dimension;


In [0]:
%sql
-- ============================================================
-- CELL 2b: Score histograms
--
-- Buckets raw scores into bands of 10 so you can see the shape
-- of the distribution at a glance. Visualise as a bar chart in
-- Databricks by switching the results view to Chart.
-- ============================================================

SELECT
  'completeness_risk'                        AS dimension,
  FLOOR(completeness_risk_score / 10) * 10   AS bucket_floor,
  COUNT(*)                                   AS people_count
FROM genealogy.gold_person_integrity_priority
GROUP BY 1, 2

UNION ALL

SELECT
  'evidence_fragility',
  FLOOR(evidence_fragility_score / 10) * 10,
  COUNT(*)
FROM genealogy.gold_person_integrity_priority
GROUP BY 1, 2

UNION ALL

SELECT
  'narrative_priority',
  FLOOR(narrative_priority_score / 10) * 10,
  COUNT(*)
FROM genealogy.gold_person_narrative_priority
GROUP BY 1, 2

ORDER BY dimension, bucket_floor;


In [0]:
%sql
-- ============================================================
-- CELL 3: Per-signal average score contribution
--
-- For each signal: how much does it actually add to scores on average
-- across the whole tree (including people where it doesn't fire)?
--
-- Formula: avg_contribution = base_score * category_weight * fire_rate
--   (approximate — ignores multipliers, but good enough for calibration)
--
-- Signals with avg_contribution < 0.5 are practically invisible in the
-- final score and probably need a higher base_score OR their guard
-- conditions reviewed.
--
-- Signals with avg_contribution > 15 may be dominating the score —
-- check whether they are truly the most important gaps.
-- ============================================================

WITH fire_rates AS (
  SELECT
    signal_code,
    COUNT(*) AS fire_count
  FROM genealogy.gold_research_person_signals_pivoted
  GROUP BY signal_code
),
total AS (
  SELECT COUNT(DISTINCT person_gedcom_id) AS total_people
  FROM genealogy.gold_research_person_signals
)
SELECT
  w.signal_code,
  w.intent,
  w.category,
  w.base_score,
  cw.weight                                               AS category_weight,
  COALESCE(fr.fire_count, 0)                             AS fire_count,
  ROUND(COALESCE(fr.fire_count, 0) * 100.0 / t.total_people, 1) AS fire_pct,
  -- Approximate average contribution to final score across all people
  ROUND(
    w.base_score
    * cw.weight
    * (COALESCE(fr.fire_count, 0) / t.total_people)
  , 2) AS avg_contribution
FROM genealogy.ref_signal_weights w
JOIN genealogy.ref_intent_category_weights cw
  ON cw.intent = w.intent AND cw.category = w.category
LEFT JOIN fire_rates fr ON fr.signal_code = w.signal_code
CROSS JOIN total t
WHERE w.base_score > 0  -- exclude SIGNAL_STORY_WRITTEN (negative weight)
ORDER BY w.intent, avg_contribution DESC;


In [0]:
%sql
-- ============================================================
-- CELL 4: Score independence cross-tab
--
-- Buckets people into high/medium/low bands for completeness and
-- evidence scores, then counts how many fall in each cell.
--
-- If completeness and evidence are measuring different things, the
-- distribution should be spread across all 9 cells.
--
-- If the matrix is diagonal (high-high / low-low only), the two
-- dimensions are measuring the same underlying thing — review
-- signal overlap.
--
-- Uses gold_person_scores (0-100 percentile-ranked scores),
-- so re-run Notebook D first if scores haven't been refreshed.
-- If Notebook D hasn't been run yet, use Cell 4b (raw scores) instead.
-- ============================================================

SELECT
  CASE
    WHEN completeness_score >= 67 THEN 'high (67-100)'
    WHEN completeness_score >= 34 THEN 'mid (34-66)'
    ELSE                               'low (0-33)'
  END AS completeness_band,
  CASE
    WHEN evidence_score >= 67 THEN 'high (67-100)'
    WHEN evidence_score >= 34 THEN 'mid (34-66)'
    ELSE                           'low (0-33)'
  END AS evidence_band,
  COUNT(*)                                    AS people_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_tree
FROM genealogy.gold_person_scores
GROUP BY 1, 2
ORDER BY 1, 2;


In [0]:
%sql
-- ============================================================
-- CELL 4b: Score independence using raw priority scores
-- Use this if Notebook D hasn't been run yet after v3.
-- Higher raw score = more problems = lower final score.
-- So 'low raw completeness_risk' = 'good completeness'.
-- ============================================================

WITH combined AS (
  SELECT
    i.person_gedcom_id,
    i.completeness_risk_score,
    i.evidence_fragility_score,
    n.narrative_priority_score
  FROM genealogy.gold_person_integrity_priority i
  JOIN genealogy.gold_person_narrative_priority n
    ON i.person_gedcom_id = n.person_gedcom_id
),
pcts AS (
  SELECT
    PERCENTILE(completeness_risk_score, 0.33)  AS comp_p33,
    PERCENTILE(completeness_risk_score, 0.67)  AS comp_p67,
    PERCENTILE(evidence_fragility_score, 0.33) AS evid_p33,
    PERCENTILE(evidence_fragility_score, 0.67) AS evid_p67
  FROM combined
)
SELECT
  CASE
    WHEN c.completeness_risk_score <= p.comp_p33 THEN 'good completeness (low risk)'
    WHEN c.completeness_risk_score <= p.comp_p67 THEN 'mid completeness'
    ELSE                                              'poor completeness (high risk)'
  END AS completeness_band,
  CASE
    WHEN c.evidence_fragility_score <= p.evid_p33 THEN 'good evidence (low fragility)'
    WHEN c.evidence_fragility_score <= p.evid_p67 THEN 'mid evidence'
    ELSE                                               'poor evidence (high fragility)'
  END AS evidence_band,
  COUNT(*) AS people_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_tree
FROM combined c
CROSS JOIN pcts p
GROUP BY 1, 2
ORDER BY 1, 2;


In [0]:
%sql
-- ============================================================
-- CELL 5: Story-ready threshold calibration
--
-- Replaces: Notebook D Cell 3c (identical query — see deduplication note)
--
-- Cross-joins a range of story_potential_score thresholds (55-90)
-- to show how many people would be flagged story-ready at each level,
-- broken down by whether they are direct ancestors.
--
-- Goal: find the threshold that gives roughly 50 story-ready people
-- with good coverage across branches and a strong direct-ancestor weighting.
--
-- Current threshold in gold_person_scores: story_potential_score >= 75
-- Currently returns 136 story-ready individuals.
-- Post-v3 this number will change — re-calibrate here before updating
-- the threshold in Notebook D.
-- ============================================================

WITH thresholds AS (
  SELECT explode(sequence(55, 90, 5)) AS threshold
),
scores AS (
  SELECT
    person_gedcom_id,
    story_potential_score,
    story_written,
    proximity_label,
    branch
  FROM genealogy.gold_person_scores
  WHERE story_written = FALSE OR story_written = 'false'
)
SELECT
  t.threshold,
  COUNT(*)                                              AS total_story_ready,
  SUM(CASE WHEN s.proximity_label = 'Direct ancestor' THEN 1 ELSE 0 END)
                                                        AS direct_ancestors,
  COUNT(DISTINCT s.branch)                              AS branches_covered,
  -- Show branch breakdown as a summary string
  CONCAT_WS(', ', COLLECT_SET(CONCAT(s.branch, ':', CAST(COUNT(s.person_gedcom_id)
    OVER (PARTITION BY t.threshold, s.branch) AS STRING))))
                                                        AS branch_counts
FROM thresholds t
JOIN scores s ON s.story_potential_score >= t.threshold
GROUP BY t.threshold
ORDER BY t.threshold;


In [0]:
%sql
-- ============================================================
-- CELL 6: Top 20 candidates per scoring dimension
--
-- Sanity check: do the people at the top of each ranking make
-- intuitive sense given what you know about your tree?
--
-- If the top completeness scorers are obscure end-of-line stubs
-- rather than direct ancestors with real gaps, the structural
-- multiplier may need tuning.
--
-- If the top story scorers are people with no primary material,
-- narrative signals may still be too speculative.
-- ============================================================

-- Top 20 by completeness score (ascending = worst completeness first)
SELECT
  'completeness' AS dimension,
  p.given_name,
  p.surname,
  p.birth_year,
  p.branch,
  s.completeness_score,
  s.evidence_score,
  s.story_potential_score,
  s.proximity_label
FROM genealogy.gold_person_scores s
JOIN genealogy.gold_person_life p ON p.person_gedcom_id = s.person_gedcom_id
ORDER BY s.completeness_score ASC
LIMIT 20;


In [0]:
%sql
-- Top 20 by story potential (descending = highest potential first)
-- These should be people with confirmed interesting angles
-- (migration, military, large family, newspaper, will/probate)
-- AND enough primary material to write from (SIGNAL_TRANSCRIPT_RICH).

SELECT
  p.given_name,
  p.surname,
  p.birth_year,
  p.branch,
  s.story_potential_score,
  s.completeness_score,
  s.evidence_score,
  s.story_ready,
  s.proximity_label,
  -- Show which narrative signals fired for this person
  CONCAT_WS(', ',
    CASE WHEN sig.SIGNAL_CONFIRMED_MILITARY  THEN 'confirmed_military'  END,
    CASE WHEN sig.SIGNAL_MIGRANT             THEN 'migrant'             END,
    CASE WHEN sig.SIGNAL_MULTIPLE_SPOUSES    THEN 'multiple_spouses'    END,
    CASE WHEN sig.SIGNAL_YOUNG_DEATH         THEN 'young_death'         END,
    CASE WHEN sig.SIGNAL_LARGE_FAMILY        THEN 'large_family'        END,
    CASE WHEN sig.SIGNAL_NEWSPAPER_MENTION   THEN 'newspaper'           END,
    CASE WHEN sig.SIGNAL_WILL_OR_PROBATE     THEN 'will_probate'        END,
    CASE WHEN sig.SIGNAL_TRANSCRIPT_RICH     THEN 'transcript_rich'     END,
    CASE WHEN sig.SIGNAL_VARIED_OCCUPATIONS  THEN 'varied_occupations'  END
  ) AS fired_narrative_signals
FROM genealogy.gold_person_scores s
JOIN genealogy.gold_person_life p   ON p.person_gedcom_id = s.person_gedcom_id
JOIN genealogy.gold_research_person_signals sig ON sig.person_gedcom_id = s.person_gedcom_id
WHERE s.story_written = FALSE OR s.story_written = 'false'
ORDER BY s.story_potential_score DESC
LIMIT 20;


In [0]:
%sql
-- ============================================================
-- CELL 7: Mutual exclusivity checks
--
-- SIGNAL_UNCONFIRMED_MILITARY and SIGNAL_CONFIRMED_MILITARY
-- should never both be TRUE for the same person.
-- If this returns any rows, check the ocr_signals CTE in Notebook B v3.
-- ============================================================

SELECT
  person_gedcom_id,
  p.given_name,
  p.surname,
  p.birth_year
FROM genealogy.gold_research_person_signals sig
JOIN genealogy.gold_person_life p ON p.person_gedcom_id = sig.person_gedcom_id
WHERE sig.SIGNAL_UNCONFIRMED_MILITARY = TRUE
  AND sig.SIGNAL_CONFIRMED_MILITARY   = TRUE;
-- Expected: 0 rows
